# RAG Evaluation: Measuring What Matters

## You've Built the Pipeline... But Does It Work?

In **Tutorial 1A**, you learned how to chunk documents.  
In **Tutorial 1B**, you learned how to optimize queries.  
In **Tutorial 2A** and **2B**, you learned retrieval methods and smart orchestration.  
In **Tutorial 3**, you learned post-retrieval context preparation.

Now comes the critical question: **How do you know any of it actually helped?**

---

## The Journey So Far

```
Tutorial 1A: Chunking ✅
    ↓
Tutorial 1B: Query Optimization ✅
    ↓
Tutorial 2A: Core Retrieval Methods ✅
    ↓
Tutorial 2B: Smart Orchestration ✅
    ↓
Tutorial 3: Post-Retrieval ✅
    ↓
Tutorial 4: Evaluation ← You are here
    ↓
Next: Agentic RAG
```

**Your pipeline runs. Evaluation tells you where it breaks.**

---

## The Challenge

A RAG system can fail at **any stage** - and a single accuracy number hides all of it:

- ✅ Retrieval returns chunks... but are they the *right* chunks?
- ✅ Generation sounds fluent... but is it *faithful* to the evidence?
- ✅ The agent picks a tool... but was it the *right* one?

**The question:** Which layer failed - retrieval, generation, tools, or the full stack?

---

## What You'll Learn

We'll walk through a **layered evaluation stack** that mirrors the pipeline you built:

**The Layers:**
1. **Retrieval Evaluation** - Hit@k, MRR, keyword overlap, RAGAS context metrics
2. **Generation Evaluation** - Faithfulness, answer relevancy, answer accuracy
3. **Tool / Agent Evaluation** - Tool selection and routing accuracy
4. **End-to-End Pipeline** - One-call scorecard across all stages

**Plus:** Baseline vs post-retrieval A/B comparison (Tutorial 3 tie-in)

**Two metric families:**
| Family | Best for |
|--------|----------|
| **Classical / Custom** | Fast, deterministic baselines (Hit@k, MRR, keyword overlap) |
| **RAGAS** | LLM-as-judge semantic metrics (faithfulness, context precision, …) |

---

## Tutorial Structure

```
Part 1: Load Test Data & Run Baseline RAG
    ↓
Part 2: Layer 1 - Retrieval Evaluation
    ↓
Part 3: Layer 2 - Generation Evaluation
    ↓
Part 4: Layer 3 - Tool / Agent Evaluation
    ↓
Part 5: End-to-End Pipeline
    ↓
Part 6: Compare Pipelines (Baseline vs Post-Retrieval)
    ↓
Part 7: Hands-On - Evaluate Your Own Query
```

**Our approach:** Score each layer independently, then read the stack as a diagnostic - not one number.

Code lives in **`retrieval_playground/src/evaluation/`**.

Let's measure what matters! 🚀


## ⚙️ Setup: Imports

We wire up the same building blocks from earlier tutorials:

| Component | Source | Role in this notebook |
|-----------|--------|----------------------|
| `RAG` | `baseline_rag.py` | Runs retrieve → generate for each test query |
| `ChunkingStrategy` | pre-retrieval | Same chunking config as tutorial 1 |
| `semantic_layer` | pre-retrieval routing (Tutorial 1B) | Demo for tool-selection evaluation in a RAG router |
| `ToolEvaluator` / `ToolTrace` | `tool_metrics.py` | Score agent-style tool decisions from logged traces |
| Evaluators | `src/evaluation/` | Score each pipeline stage |

> **Note:** Logging is suppressed so live demo output stays readable during presentation.


In [4]:
import retrieval_playground.utils.ragas_compat

import json
import warnings
from typing import Dict, List, Any

import pandas as pd

from retrieval_playground.utils import config
from retrieval_playground.src.baseline_rag import RAG
from retrieval_playground.src.pre_retrieval.routing import semantic_layer
from retrieval_playground.src.evaluation import (
    RetrievalEvaluator,
    GenerationEvaluator,
    ToolEvaluator,
    ToolTrace,
    RAGEvaluationPipeline,
)

import logging
warnings.filterwarnings('ignore')
logging.disable(logging.CRITICAL)

## 📂 Load Test Data & Run Baseline RAG

Every evaluation metric needs a **ground truth** to compare against. Our test set (`evaluation_dataset.json`) provides two gold labels per query:

| Field | What it is | Example use |
|-------|------------|---------------|
| `user_input` | The question | Drives retrieval & generation |
| `reference_context` | Gold **evidence** passage | Classical retrieval metrics |
| `reference` | Gold **answer** | Generation metrics & RAGAS context metrics |

> ⚠️ **Important distinction:** Classical retrieval metrics compare chunks → `reference_context`. RAGAS context_precision / context_recall compare chunks → `reference` (the answer). Mixing these up gives misleading scores.

We run baseline RAG on a **small slice** (3 queries) so live demos finish quickly - swap in the full test set for production benchmarking.


In [5]:
def load_test_queries() -> List[Dict[str, Any]]:
    queries_path = config.TEST_DATA_DIR / "evaluation_dataset.json"
    with open(queries_path, "r") as f:
        return json.load(f)

strategy = "recursive_character"
rag = RAG(strategy=strategy)

# Use a small slice for the tutorial demo
test_queries = load_test_queries()[:3]
questions = [q["user_input"] for q in test_queries]
ground_truths = [q["reference"] for q in test_queries]
reference_contexts = [q["reference_context"] for q in test_queries]

rag_results = []
retrieved_contexts = []

for query in test_queries:
    result = rag.query(query["user_input"])
    rag_results.append(result)
    retrieved_contexts.append([ctx["content"] for ctx in result["context"]])

answers = [r["answer"] for r in rag_results]

print(f"Evaluating {len(test_queries)} queries")
print(f"Sample: {questions[0][:80]}...")

Evaluating 3 queries
Sample: why build a climate KG for AutoClimDS — can't frontier LLMs just web-search CMIP...


---

# Layer 1: Retrieval Evaluation

## The Question

> *"Did the retriever surface evidence that actually supports the answer?"*

If retrieval fails, **no amount of prompt engineering fixes generation**. This layer catches problems before they reach the LLM.

---

## Metrics at a Glance

| Metric | Type | Range | What it tells you | Low score means… |
|--------|------|-------|-------------------|------------------|
| **Hit Rate @ k** | Classical | 0–1 | Any relevant chunk in top-k? | Retriever never surfaces gold evidence |
| **MRR** | Classical | 0–1 | How early does the first relevant chunk appear? | Right chunk exists but ranks too low |
| **Keyword Overlap** | Custom (lexical) | 0–1 | Token overlap with gold evidence | Semantic match without lexical overlap |
| **Context Precision** | RAGAS | 0–1 | Fraction of retrieved chunks that matter | Too much noise in the context window |
| **Context Recall** | RAGAS | 0–1 | Fraction of needed evidence retrieved | Gold facts missing from retrieval |

---

See **`retrieval_metrics.py`** and **`ragas_runner.py`** for implementations.

In [6]:
# Takes approx 3 minutes

retrieval_evaluator = RetrievalEvaluator(k=3)

# Classical metrics — compare chunks to gold evidence (reference_context)
classical = retrieval_evaluator.evaluate(
    questions=questions,
    contexts=retrieved_contexts,
    reference_contexts=reference_contexts,
)
print("=== Classical Retrieval Metrics ===")
for name, score in classical.scores.items():
    print(f"  {name}: {score:.3f}")

# RAGAS retrieval metrics — compare chunks to gold ANSWERS (reference)
ragas_retrieval = retrieval_evaluator.evaluate_ragas(
    questions=questions,
    contexts=retrieved_contexts,
    reference_answers=ground_truths,
    answers=answers,
)
print("\n=== RAGAS Retrieval Metrics (vs reference answers) ===")
for name, score in ragas_retrieval.items():
    print(f"  {name}: {score:.3f}")

=== Classical Retrieval Metrics ===
  hit_rate_at_k@3: 1.000
  mrr: 1.000
  keyword_overlap: 0.982


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]


=== RAGAS Retrieval Metrics (vs reference answers) ===
  context_precision: 1.000
  context_recall: 1.000


---

# Layer 2: Generation Evaluation

## The Question

> *"Given the retrieved context, is the answer actually good?"*

Retrieval can be perfect and generation can still fail - by hallucinating facts, ignoring the question, or being uselessly verbose.

---

## Metrics at a Glance

| Metric | Type | What it measures | Low score means… |
|--------|------|------------------|------------------|
| **Faithfulness** | RAGAS | Is every claim in the answer supported by context? | Model invented facts not in the documents |
| **Answer Relevancy** | RAGAS | Does the answer address what was asked? | Correct info, wrong question |
| **Answer Accuracy** | RAGAS (NVIDIA) | Does the answer match the gold reference? | Right topic but wrong or incomplete facts |
| **Answer Length Ratio** | Custom | Is the answer proportionally concise vs gold? | Too verbose or too terse |

---

See **`generation_metrics.py`** for implementations.


In [4]:
# Takes approx 7 minutes

generation_evaluator = GenerationEvaluator(
    ragas_metrics=["faithfulness", "answer_relevancy", "answer_accuracy"]
)
generation_result = generation_evaluator.evaluate(
    questions=questions,
    answers=answers,
    contexts=retrieved_contexts,
    ground_truths=ground_truths,
)

print("=== RAGAS Generation Metrics ===")
for name in ["faithfulness", "answer_relevancy", "answer_accuracy"]:
    if name in generation_result.ragas_scores:
        print(f"  {name}: {generation_result.ragas_scores[name]:.3f}")

print("\n=== Custom Generation Metrics ===")
for name, score in generation_result.custom_scores.items():
    print(f"  {name}: {score:.3f}")

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

=== RAGAS Generation Metrics ===
  faithfulness: 0.987
  answer_relevancy: 0.910
  answer_accuracy: 0.750

=== Custom Generation Metrics ===
  answer_length_ratio: 10.039
410.57243251800537


## The Failure Modes

| Symptom | Likely culprit | Metric to check | Where to fix |
|---------|---------------|-----------------|--------------|
| Answer cites facts not in docs | Hallucination | ↓ Faithfulness | Post-retrieval grading, stricter prompts |
| Answer is factually fine but misses the point | Relevancy gap | ↓ Answer Relevancy | Query rewriting, better retrieval |
| Answer sounds good but disagrees with gold | Factual mismatch | ↓ Answer Accuracy | Retrieval coverage, model choice |
| Answer is a wall of text | Verbosity | ↑ Length Ratio | Context compression (Tutorial 3) |

> 💡 **Presentation tip:** Faithfulness, answer relevancy, and answer accuracy are *independent*. A model can be faithful (no hallucination) but irrelevant (answers a different question) or accurate-looking yet unfaithful to the retrieved context.

---

# Layer 3: Tool / Agent Evaluation

## The Question

> *"Did the system pick the right tool, execute it successfully, and use the result appropriately?"*

Modern RAG is rarely a single retrieve → generate path. **Agents** break a user request into steps, choosing among tools at each step - vector search, SQL, web search, calculators, code interpreters, and more. A wrong tool choice upstream poisons everything downstream, even when retrieval and generation metrics look fine.

---

## Tool Use in Agents: What Actually Happens

In frameworks like ReAct, Toolformer, or LangGraph-style agents, the loop typically looks like:

```
User query → [Plan] → select tool → pass arguments → execute → observe result → (repeat) → final answer
```

Each step is a **decision point** you can evaluate independently:

| Decision | Example failure | What you'd log |
|----------|-----------------|----------------|
| **Which tool?** | Web search for a question answerable from your vector DB | `expected_tool` vs `actual_tool` |
| **Which variant?** | Dense search when the query needs multi-query expansion | `expected_method` vs `actual_method` |
| **Did it run?** | API timeout, bad credentials, malformed args | `success: true/false` |
| **Were args right?** | SQL with wrong table, search with empty filter | Schema validation, arg diff |
| **Was output used?** | Tool returned data but the agent ignored it | Downstream faithfulness check |

> **Scope in this repo:** `ToolEvaluator` scores the first three columns via `ToolTrace` objects. Argument correctness and output utilization are natural extensions (schema checks, LLM judges) - the trace format is designed to grow into those.

---

### Metrics at a Glance

| Metric | What it measures | Low score means… |
|--------|------------------|------------------|
| **Tool Selection Accuracy** | Correct tool chosen? | Agent hits wrong data source or skips retrieval when needed |
| **Retrieval Method Accuracy** | Correct search strategy within a tool? | Right tool, wrong tactic (e.g. hybrid vs multi-query) |
| **Tool Call Success Rate** | Did the invocation complete? | Infra failures, bad args, or API errors |

---

## Demo: Semantic Router from Tutorial 1B

Our live demo uses the **semantic routing layer** from Tutorial 1B (`routing.py`) - a concrete, lightweight example of tool selection in RAG. The router is not a full multi-step agent, but it makes the same core decision: *given this query, which tool (and retrieval method) should run?*

That maps directly onto agent evaluation:

| Tutorial 1B concept | Agent eval concept |
|--------------------|--------------------|
| `semantic_layer(query)` | Tool-selection step in an agent loop |
| `vector_db` / `llm_direct` / `web_search` | Tool registry entries |
| `hybrid_search` / `multi_query` | Sub-tool or method variant |
| `ROUTE_METADATA` | Labeled gold traces for benchmarking |

We evaluate against **labeled routing cases** - each query has an expected tool and retrieval method:

| Query type | Expected tool | Expected method |
|------------|---------------|-----------------|
| Factual question | `vector_db` | `hybrid_search` |
| Comparison / multi-facet | `vector_db` | `multi_query` |
| Greeting / chitchat | `llm_direct` | *(none)* |

> 🔧 **Fix playbook:**
- Low tool selection accuracy → tune the router or tool descriptions.
- Low method accuracy → refine sub-strategy rules.
- Low success rate → check API health and argument schemas before blaming the LLM.

See **`tool_metrics.py`** for `ToolEvaluator`, `ToolTrace`, and `build_traces_from_routing()`.


In [5]:
# Labeled routing cases: expected tool/method for sample queries
# (aligned with ROUTE_METADATA in routing.py)
routing_cases = [
    {
        "query": "What is chain-of-thought prompting?",
        "expected_tool": "vector_db",
        "expected_method": "hybrid_search",
    },
    {
        "query": "Compare RAG and fine-tuning approaches",
        "expected_tool": "vector_db",
        "expected_method": "multi_query",
    },
    {
        "query": "Hello, how are you?",
        "expected_tool": "llm_direct",
        "expected_method": None,
    },
]

tool_evaluator = ToolEvaluator()
tool_traces = ToolEvaluator.build_traces_from_routing(routing_cases, semantic_layer)
tool_result = tool_evaluator.evaluate(tool_traces)

print("=== Tool / Routing Metrics ===")
for name, score in tool_result.scores.items():
    print(f"  {name}: {score:.3f}")

print("\n=== Per-Query Routing Decisions ===")
for trace in tool_traces:
    match = "✅" if trace.actual_tool == trace.expected_tool else "❌"
    print(f"  {match} {trace.query[:50]}... → {trace.actual_tool} (expected: {trace.expected_tool})")

=== Tool / Routing Metrics ===
  tool_selection_accuracy: 1.000
  retrieval_method_accuracy: 0.500
  tool_call_success_rate: 0.667

=== Per-Query Routing Decisions ===
  ✅ What is chain-of-thought prompting?... → vector_db (expected: vector_db)
  ✅ Compare RAG and fine-tuning approaches... → vector_db (expected: vector_db)
  ✅ Hello, how are you?... → llm_direct (expected: llm_direct)


---

# Layer 4: Custom Metrics

## Why Extend Beyond Built-ins?

Off-the-shelf metrics cover general RAG quality, but production systems often need **domain-specific signals** - regulatory citations, token budgets, mandatory keywords, etc.

**Lightweight helpers** live in **`base.py`**:

```python
from retrieval_playground.src.evaluation.base import keyword_overlap, answer_length_ratio
```

For RAGAS-native custom metrics, subclass `SingleTurnMetric` - see the [RAGAS custom metrics guide](https://docs.ragas.io/en/stable/howtos/customizations/metrics/_write_your_own_metric/).


In [6]:
from retrieval_playground.src.evaluation.base import keyword_overlap, answer_length_ratio

print(f"keyword_overlap (sample): {keyword_overlap(retrieved_contexts[0], reference_contexts[0]):.3f}")
print(f"answer_length_ratio (sample): {answer_length_ratio(answers[0], ground_truths[0]):.3f}")

keyword_overlap (sample): 1.000
answer_length_ratio (sample): 9.000


### RAGAS custom metric template

```python
from dataclasses import dataclass, field
from ragas.metrics.base import SingleTurnMetric, MetricType

@dataclass
class MyRagasMetric(SingleTurnMetric):
    name: str = "my_ragas_metric"
    _required_columns: dict = field(default_factory=lambda: {
        MetricType.SINGLE_TURN: {"user_input", "response"}
    })

    async def _single_turn_ascore(self, sample, callbacks) -> float:
        return 1.0  # your scoring logic
```

| Design choice | Recommendation |
|---------------|----------------|
| Metric name | Use `snake_case`, unique across your eval suite |
| Return value | Float in **0.0–1.0** range for consistency with RAGAS |
| Batch vs single | Use `base.py` helpers for batch heuristics; RAGAS calls `_single_turn_ascore()` per sample |


---

# 🔗 End-to-End Evaluation Pipeline

## One Call, All Stages

`RAGEvaluationPipeline` orchestrates retrieval, generation, and tool evaluators in a single pass - producing a unified scorecard you can export, plot, or compare across experiments.

```
Input (queries, answers, contexts, ground truths, tool traces)
    → RetrievalEvaluator  →  retrieval scores
    → GenerationEvaluator →  generation scores
    → ToolEvaluator       →  tool scores (if traces provided)
    → Summary DataFrame   →  stage · metric · score
```

| Output column | Meaning |
|---------------|---------|
| `stage` | `retrieval`, `generation`, or `tools` |
| `metric` | Metric name (e.g. `hit_rate_at_k@3`, `faithfulness`) |
| `score` | Aggregated score across the batch |

> **Live demo moment:** Run the cell below and scan the table - which stage is the weakest link for your current RAG config?


In [7]:
# Takes approx 8 minutes

pipeline = RAGEvaluationPipeline(
    retrieval_k=2,
    ragas_metrics=["faithfulness", "answer_relevancy", "answer_accuracy"],
)

pipeline_result = pipeline.evaluate_rag_results(
    rag_results=rag_results,
    ground_truths=ground_truths,
    reference_contexts=reference_contexts,
    tool_traces=tool_traces,
)

summary_df = pipeline_result.to_dataframe()
print("=== Full Pipeline Evaluation ===")
print(summary_df.to_string(index=False))

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

=== Full Pipeline Evaluation ===
     stage                    metric     score
 retrieval           hit_rate_at_k@2  1.000000
 retrieval                       mrr  1.000000
 retrieval           keyword_overlap  0.981982
 retrieval         context_precision  1.000000
 retrieval            context_recall  1.000000
generation              faithfulness  0.987179
generation          answer_relevancy  0.912604
generation           answer_accuracy  1.000000
generation       answer_length_ratio 10.038889
     tools   tool_selection_accuracy  1.000000
     tools retrieval_method_accuracy  0.500000
     tools    tool_call_success_rate  0.666667
485.46090960502625


## ⚖️ Compare Pipelines: Baseline vs Post-Retrieval

This is the canonical place to **score** whether Tutorial 3's context preparation actually helped. We run the same queries through two paths and compare metrics from every layer above.

| Pipeline | Flow |
|----------|------|
| **A - Baseline** | Retrieve → `document_assembly.generate_answer()` → score |
| **B - Post-retrieval** | Retrieve → grade · refine · compress → `document_assembly.generate_answer()` → score |

Both arms use the **same generator** so score differences reflect context preparation, not a different prompt chain.

### How to read surprising scores

**Context recall ≈ 0 for both rows** - RAGAS recall checks claim coverage against the gold *reference answer*, not `reference_context`. With small `k` and aggressive compression, recall often stays at zero even when answers look fine.

**Baseline beats post-retrieval on answer accuracy** - Compression removes text the model no longer sees. Shorter context can mean less evidence, even if it is cleaner.

**Precision up, recall down** - Common when trimming noise: you traded breadth for focus. Check operational stats (chunks and tokens in → out) printed in the cell below.


In [7]:
# takes approx 12 minutes

from retrieval_playground.src.post_retrieval import ContextPreparer, document_assembly
from retrieval_playground.src.evaluation import RAGEvaluator

CONTEXT_METRICS = ["context_precision", "context_recall"]
GEN_METRICS = ["faithfulness", "answer_relevancy", "answer_accuracy"]

comparison_queries = test_queries[:2]  # smaller slice for A/B demo
comparison_ground_truths = [q["reference"] for q in comparison_queries]

preparer = ContextPreparer()
baseline_rag_results = []
prepared_rag_results = []

for query_data in comparison_queries:
    query = query_data["user_input"]
    retrieved = [doc for doc, _ in rag.retrieve_context(query, k=2)]

    baseline_answer = document_assembly.generate_answer(query, retrieved)
    baseline_rag_results.append({
        "question": query,
        "answer": baseline_answer,
        "context": [{"content": doc.page_content} for doc in retrieved],
    })

    prepared = preparer.prepare(query, retrieved)
    prepared_answer = document_assembly.generate_answer(query, prepared.chunks)
    prepared_rag_results.append({
        "question": query,
        "answer": prepared_answer,
        "context": [{"content": doc.page_content} for doc in prepared.chunks],
    })

    print(f"{query[:70]}...")
    print(f"  chunks {len(retrieved)} → {len(prepared.chunks)} | tokens {prepared.token_before} → {prepared.token_after}")

context_evaluator = RAGEvaluator(metrics=CONTEXT_METRICS)
gen_evaluator = GenerationEvaluator(ragas_metrics=GEN_METRICS)

baseline_context = context_evaluator.evaluate_rag_results(baseline_rag_results, comparison_ground_truths)
prepared_context = context_evaluator.evaluate_rag_results(prepared_rag_results, comparison_ground_truths)
baseline_gen = gen_evaluator.evaluate_rag_results(baseline_rag_results, comparison_ground_truths).scores
prepared_gen = gen_evaluator.evaluate_rag_results(prepared_rag_results, comparison_ground_truths).scores

metric_names = CONTEXT_METRICS + GEN_METRICS + ["answer_length_ratio"]
rows = []
for metric in metric_names:
    if metric in CONTEXT_METRICS:
        baseline_val = baseline_context[metric]
        prepared_val = prepared_context[metric]
    else:
        baseline_val = baseline_gen[metric]
        prepared_val = prepared_gen[metric]
    rows.append({
        "Metric": metric,
        "Baseline": round(baseline_val, 3),
        "Post-Retrieval Prepared": round(prepared_val, 3),
    })

comparison_df = pd.DataFrame(rows)
print("\n=== Baseline vs Post-Retrieval ===")
comparison_df

why build a climate KG for AutoClimDS — can't frontier LLMs just web-s...
  chunks 2 → 2 | tokens 522 → 189
can chatgpt-4 speed up pillow/numpy internals without a maintainer in ...
  chunks 2 → 2 | tokens 615 → 148


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]


=== Baseline vs Post-Retrieval ===
755.7559781074524


,Metric,Baseline,Post-Retrieval Prepared
0,context_precision,1.000,1.000
1,context_recall,1.000,0.500
2,faithfulness,0.857,0.938
3,answer_relevancy,0.879,0.924
4,answer_accuracy,0.875,0.875
5,answer_length_ratio,6.029,4.300


## Classical vs RAGAS Retrieval Metrics

```
┌─────────────────────────────────────────────────────────────┐
│  CLASSICAL (fast, deterministic)                            │
│  Compare: retrieved chunks  ↔  reference_context (evidence) │
│  Metrics: Hit@k, MRR, keyword_overlap                       │
├─────────────────────────────────────────────────────────────┤
│  RAGAS (LLM-judged, semantic)                               │
│  Compare: retrieved chunks  ↔  reference (gold answer)      │
│  Metrics: context_precision, context_recall                 │
└─────────────────────────────────────────────────────────────┘
```

> 🔧 **Fix playbook:**
 - Low Hit@k → try hybrid search or better chunking (Tutorial 2)
 - Low MRR → add reranking
 - Low context recall → increase k or improve embeddings


---

## Hands-On: Evaluate Your Own Query

Everything above runs on a fixed test slice. This section lets you **point the full evaluation stack at any query** during a live demo.

**How to use during presentation:**

1. Uncomment the example call at the bottom of the cell
2. Swap in your own question and gold labels
3. Read the scores - which stage failed?

| Score pattern | Diagnosis |
|---------------|-----------|
| Low Hit@k, good faithfulness | Retriever missed evidence; generation made best of bad context |
| Good Hit@k, low faithfulness | Retrieved well but model hallucinated anyway |
| Good faithfulness, low answer accuracy | Grounded in context but still wrong vs gold reference |
| Good everything, low relevancy | Answer is grounded but doesn't address the question |


In [ ]:
def evaluate_query(query: str, reference: str, reference_context: str):
    """End-to-end eval for a single custom query."""
    result = rag.query(query)
    contexts = [[ctx["content"] for ctx in result["context"]]]

    retrieval = RetrievalEvaluator(k=3).evaluate(
        [query], contexts, [reference_context]
    )
    generation = GenerationEvaluator().evaluate(
        [query], [result["answer"]], contexts, [reference]
    )

    print(f"Query: {query}\n")
    print("Retrieval:", {k: round(v, 3) for k, v in retrieval.scores.items()})
    print("Generation:", {k: round(v, 3) for k, v in generation.scores.items()})
    print(f"\nAnswer: {result['answer'][:300]}...")

# Example
evaluate_query(
    test_queries[1]["user_input"],
    test_queries[1]["reference"],
    test_queries[1]["reference_context"],
)

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Query: why build a climate KG for AutoClimDS — can't frontier LLMs just web-search CMIP datasets?

Retrieval: {'hit_rate_at_k@3': 1.0, 'mrr': 1.0, 'keyword_overlap': 0.84}
Generation: {'faithfulness': 1.0, 'answer_relevancy': 0.806, 'answer_accuracy': 1.0, 'answer_length_ratio': 9.886}

Answer: Based on the provided context, a climate Knowledge Graph (KG) was built for AutoClimDS because frontier Large Language Models (LLMs) with web search capabilities alone are not sufficient for autonomously and reliably performing scientific workflows with climate datasets.

Here is a comprehensive bre...


---

## 🎯 Conclusion: Evaluation Is a Diagnostic Stack


### What We Covered

| Layer | Key metrics | Module |
|-------|-------------|--------|
| Retrieval | Hit@k, MRR, context precision/recall | `retrieval_metrics.py` |
| Generation | Faithfulness, answer relevancy, answer accuracy, length ratio | `generation_metrics.py` |
| Tools / Agents | Tool selection, method accuracy, call success | `tool_metrics.py` |
| Custom | Your domain-specific checks | `base.py` helpers + RAGAS `SingleTurnMetric` |
| Pipeline | Unified scorecard | `pipeline.py` |

### Key Takeaways

1. **Diagnose by stage** - a low aggregate score tells you nothing; a low *stage* score tells you where to invest engineering time.
2. **Use classical metrics for speed, RAGAS for nuance** - run cheap checks in CI, run LLM-judged metrics on release candidates.
3. **Compare pipelines, not just scores** - the A/B table (baseline vs post-retrieval) is how you prove optimizations work.
4. **Extend with custom metrics** - when off-the-shelf metrics miss your domain, use `base.py` helpers for fast heuristics or subclass RAGAS `SingleTurnMetric` for LLM-judged checks.

### Further Reading

- [RAGAS metrics reference](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)
- Custom metrics guide: `src/evaluation/EVALUATION.md`
- Prior tutorials: chunking (1) → retrieval (2) → post-retrieval (3) → **evaluation (this notebook)**
